In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from IPython.display import display, Markdown, Math, HTML

import numpy as np
from scipy.constants import epsilon_0, c, hbar
from pyGDM2 import fields

from optomechanics_levitation import particle_properties, rotate_particle, evaluate_incident_field, evaluate_internal_field, evaluate_far_field

In [35]:
## PARTICLE PROPERTIES
long_axis = 100.0; long_radius = long_axis / 2.0; aspect_ratio = 2.0; short_radius = long_radius / aspect_ratio # [nm]
wavelength = 1550.0 # [nm]
intensity = 1.0e9 # [W/m^2]

# Discretization step
if aspect_ratio == 2.0:
    mesh_divisions = 35.0
else:
    mesh_divisions = 65.0
step = long_axis / mesh_divisions # [nm]

# x-oriented ellipsoidal particle
ellipsoidal_particle = particle_properties(particle_type="spheroid", material_name="sio2", step_nm=step, mesh="cube",
                                           center=True, R1=long_radius / step - 0.2, R2=short_radius / step - 0.2, R3=short_radius / step - 0.2)

# INTERACTION PARAMETERS
far_field_radius = 1e6 # [nm]

In [36]:
def militaru_torque_recoil_heating(long_radius_nm, short_radius_nm, epsilon_r, intensity, wavelength):
    a = long_radius_nm * 1e-9; b = short_radius_nm * 1e-9
    e = np.sqrt(1 - (b / a)**2)

    L_x = (1 - e**2) / (2 * e**3) * (np.log((1 + e) / (1 - e)) - 2 * e)
    L_y = (1 - L_x) / 2
    volume = 4 * np.pi * a * b**2 / 3

    alpha_x = epsilon_0 * volume * (epsilon_r - 1) / (1 + L_x * (epsilon_r - 1))
    alpha_y = epsilon_0 * volume * (epsilon_r - 1) / (1 + L_y * (epsilon_r - 1))
    delta_alpha = alpha_x - alpha_y

    E0 = np.sqrt(2 * intensity / (epsilon_0 * c))
    omega0 = 2 * np.pi * c / (wavelength * 1e-9)

    P_sc = omega0**4 * alpha_x**2 * E0**2 / (12 * np.pi * epsilon_0 * c**3)
    Gamma_sc = P_sc / (hbar * omega0)
    S_tau_tau = hbar**2 * Gamma_sc / (2 * np.pi) * (delta_alpha / alpha_x)**2

    return Gamma_sc, S_tau_tau

gamma_m_sc, torque_noise_militaru = militaru_torque_recoil_heating(long_radius, short_radius, ellipsoidal_particle["material"].epsilon(wavelength).real, intensity, wavelength)

In [37]:
environment = {"eps1": 1.0, "eps2": 1.0, "eps3": 1.0, "spacing": 0.0}
E0 = np.sqrt(2 * intensity / (epsilon_0 * c))

incident_parameters = dict(wavelength_nm=wavelength, environment=environment, amplitude_v_per_m=E0, inc_angle=0.0, inc_plane="xz", E_p=1.0, E_s=0.0)
incident_field = evaluate_incident_field(field_generator=fields.plane_wave, positions_nm=ellipsoidal_particle["geometry_nm"], field="E", **incident_parameters)
interaction = evaluate_internal_field(particle=ellipsoidal_particle, incident_electric_field=incident_field, wavelength_nm=wavelength, environment=environment)

In [38]:
from optomechanics_levitation import force_spectral_density, torque_spectral_density

In [39]:
force = force_spectral_density(interaction, fields.plane_wave, incident_parameters, force_axis="y")
torque_noise_simulation, spin_term, orbital_term, mixed_term = torque_spectral_density(interaction, fields.plane_wave, incident_parameters, torque_axis="y",  n_theta=64, n_phi=64)

In [40]:
torque_noise_simulation = torque_noise_simulation / (2 * np.pi)

# Results 

In [41]:
## RELATIVE DIFFERENCES
relative_difference = (np.abs(torque_noise_simulation - torque_noise_militaru) / torque_noise_militaru) * 100

In [42]:
def sci_notation(value, decimals):
    """ Function to write values in scientific notation """
    mantissa, exponent = f"{value:.{decimals}e}".split("e")
    return rf"{mantissa} \times 10^{{{int(exponent)}}}" 

display(Markdown(rf"""

$S_{{\tau \tau}}^{{\mu, M}} = {sci_notation(torque_noise_militaru, 2)} \,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$

$S_{{\tau \tau}}^{{\mu, sim}} = {sci_notation(torque_noise_simulation, 2)} \,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$

$\text{{Relative difference}} = {relative_difference:,.2f}\%$

$S_{{\mu, \hat{{S}}}} = {sci_notation(spin_term, 2)} \,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$

$S_{{\mu, \hat{{L}}}} = {sci_notation(orbital_term, 2)} \,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$

$S_{{\mu, \hat{{S}} \hat{{L}}}} = {sci_notation(mixed_term, 2)} \,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$

"""))



$S_{\tau \tau}^{\mu, M} = 9.11 \times 10^{-62} \,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$

$S_{\tau \tau}^{\mu, sim} = 9.21 \times 10^{-62} \,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$

$\text{Relative difference} = 1.02\%$

$S_{\mu, \hat{S}} = 7.18 \times 10^{-60} \,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$

$S_{\mu, \hat{L}} = 4.55 \times 10^{-60} \,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$

$S_{\mu, \hat{S} \hat{L}} = -5.58 \times 10^{-60} \,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$

